In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (they are in the current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Aggregate metrics
metrics_grp = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_points=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        global_p95_value=('value', lambda x: x.quantile(0.95))
    )
)

# Convert min/max timestamps to timezone-aware datetimes (UTC -> Asia/Shanghai)
metrics_grp['min_timestamp'] = pd.to_datetime(metrics_grp['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metrics_grp['max_timestamp'] = pd.to_datetime(metrics_grp['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Sort and limit to top 50 by count_of_points
metrics_agg = metrics_grp.sort_values(['count_of_points', 'cmdb_id'], ascending=[False, True]).head(50)

# Tidy columns and round p95 for compactness
metrics_agg['global_p95_value'] = metrics_agg['global_p95_value'].astype(float).round(6)
metrics_agg = metrics_agg[['cmdb_id', 'kpi_name', 'count_of_points', 'min_timestamp', 'max_timestamp', 'global_p95_value']]

# Aggregate traces
traces_grp = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_of_points=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        global_p95_value=('value', lambda x: x.quantile(0.95))
    )
)

traces_grp['min_timestamp'] = pd.to_datetime(traces_grp['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
traces_grp['max_timestamp'] = pd.to_datetime(traces_grp['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

traces_agg = traces_grp.sort_values(['count_of_points', 'cmdb_id'], ascending=[False, True]).head(50)
traces_agg['global_p95_value'] = traces_agg['global_p95_value'].astype(float).round(6)
traces_agg = traces_agg[['cmdb_id', 'trace_name', 'count_of_points', 'min_timestamp', 'max_timestamp', 'global_p95_value']]

# Display compact aggregated tables: metrics first, then traces
metrics_agg, traces_agg

```
Out[1]:
```
# Build a concise plain-English summary string using the already-computed aggregates
top_m = metrics_agg[['cmdb_id','kpi_name','count_of_points','global_p95_value','min_timestamp','max_timestamp']].head(5)
top_t = traces_agg[['cmdb_id','trace_name','count_of_points','global_p95_value','min_timestamp','max_timestamp']].head(5)

lines = []
lines.append(f"Metrics aggregate: returned {metrics_agg.shape[0]} groups (top 50 by point count).")
for r in top_m.itertuples(index=False):
    lines.append(f"- {r.cmdb_id} | {r.kpi_name} — {r.count_of_points} pts, p95={float(r.global_p95_value):.6f}, range={r.min_timestamp} to {r.max_timestamp}")

lines.append("")    
lines.append(f"Traces aggregate: returned {traces_agg.shape[0]} groups (top 50 by point count).")
for r in top_t.itertuples(index=False):
    lines.append(f"- {r.cmdb_id} | {r.trace_name} — {r.count_of_points} pts, p95={float(r.global_p95_value):.6f}, range={r.min_timestamp} to {r.max_timestamp}")

lines.append("")
lines.append("Notes: many metric groups have 61 or 60 points spanning 2020-05-25 04:00+08 to 04:59/05:00+08; many trace groups have 30 points spanning 2020-05-25 04:30+08 to 04:59+08. Full aggregated tables are in variables `metrics_agg` and `traces_agg`.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id             kpi_name  count_of_points             min_timestamp             max_timestamp  global_p95_value
5    db_001         CPU_free_pct               61 2020-05-25 04:00:00+08:00 2020-05-25 05:00:00+08:00         98.901685
95   db_003         CPU_free_pct               61 2020-05-25 04:00:00+08:00 2020-05-25 05:00:00+08:00         99.159329
289  db_007        MEM_real_util               61 2020-05-25 04:00:00+08:00 2020-05-25 05:00:00+08:00         81.930000
322  db_008         CPU_free_pct               61 2020-05-25 04:00:00+08:00 2020-05-25 05:00:00+08:00         97.637021
955  os_006           Send_total               61 2020-05-25 04:00:00+08:00 2020-05-25 05:00:00+08:00          0.000476
..      ...                  ...              ...                       ...                       ...               ...
44   db_001  tnsping_result_time               60 2020-05-25 04:00:00+08:00 2020-05-25 04:59:00+08:00         10.000000
45   db_002                  ACS               60 2020-05-25 04:00:00+08:00 2020-05-25 04:59:00+08:00          0.020000
46   db_002                 AIOS               60 2020-05-25 04:00:00+08:00 2020-05-25 04:59:00+08:00          0.000000
47   db_002                  AWS               60 2020-05-25 04:00:00+08:00 2020-05-25 04:59:00+08:00          0.020000
50   db_002         CPU_free_pct               60 2020-05-25 04:00:00+08:00 2020-05-25 04:59:00+08:00         99.205579

[50 rows x 6 columns],        cmdb_id                                       trace_name  count_of_points             min_timestamp             max_timestamp  global_p95_value
0   docker_001    trace.from_os_022.remoteprocess.duration_mean               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.968829
1   docker_001     trace.from_os_022.remoteprocess.duration_p95               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          2.504855
2   docker_001       trace.from_os_022.remoteprocess.error_rate               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00        145.550000
4   docker_001                     trace.self.csf.duration_mean               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.086022
..         ...                                              ...              ...                       ...                       ...               ...
45  docker_002                    trace.self.local.duration_p95               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.097000
46  docker_002                      trace.self.local.error_rate               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.000000
47  docker_002                       trace.self.local.row_count               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       2082.500000
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.045960
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95               30 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.057685

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global thresholds (p95 and p05) per (cmdb_id, kpi_name) using full series
metrics_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_points=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        global_p95=('value', lambda x: float(x.quantile(0.95))),
        global_p05=('value', lambda x: float(x.quantile(0.05)))
    )
)

# convert min/max timestamps to timezone-aware datetimes (UTC -> Asia/Shanghai)
metrics_agg['min_timestamp'] = pd.to_datetime(metrics_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metrics_agg['max_timestamp'] = pd.to_datetime(metrics_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Filter raw metric points to incident window (inclusive)
df_metric_ts = df_metric.copy()
df_metric_ts['ts'] = pd.to_datetime(df_metric_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

window_start = pd.Timestamp('2020-05-25 04:30:00', tz=tz)
window_end = pd.Timestamp('2020-05-25 05:00:00', tz=tz)

df_window = df_metric_ts[(df_metric_ts['ts'] >= window_start) & (df_metric_ts['ts'] <= window_end)].copy()

# Merge thresholds into window points
df_win_merged = df_window.merge(metrics_agg[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')

# Identify anomalies: value >= p95 => high; value <= p05 => low
cond_high = df_win_merged['value'] >= df_win_merged['global_p95']
cond_low = df_win_merged['value'] <= df_win_merged['global_p05']
df_win_merged['flagged_side'] = np.where(cond_high, 'high', np.where(cond_low, 'low', None))

# Keep only anomaly rows
anomalies_all = df_win_merged[df_win_merged['flagged_side'].notnull()].copy()

# Add used_threshold and threshold_value columns
anomalies_all['used_threshold'] = np.where(anomalies_all['flagged_side']=='high', 'p95', 'p05')
anomalies_all['threshold_value'] = np.where(anomalies_all['flagged_side']=='high', anomalies_all['global_p95'], anomalies_all['global_p05'])

# Compute per-row severity: (value - p95)/p95 for high, (p05 - value)/p05 for low. Guard divide-by-zero.
def compute_severity(row):
    if row['flagged_side'] == 'high':
        th = row['global_p95']
        return (row['value'] - th) / th if th != 0 else np.nan
    else:
        th = row['global_p05']
        return (th - row['value']) / th if th != 0 else np.nan

anomalies_all['severity_row'] = anomalies_all.apply(compute_severity, axis=1)
anomalies_all['severity_row'] = anomalies_all['severity_row'].abs()

# 3) Produce compact summary top 20 by anomaly_count
grouped = (
    anomalies_all
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomaly_count=('value', 'size'),
        earliest_anomaly_timestamp=('ts', 'min'),
        latest_anomaly_timestamp=('ts', 'max'),
        max_anomalous_value=('value', 'max'),
        min_anomalous_value=('value', 'min'),
        severity_metric=('severity_row', 'max')  # already positive
    )
)

# Sort and keep top 20
top20_summary = grouped.sort_values('anomaly_count', ascending=False).head(20).copy()

# Round severity metric for compactness
top20_summary['severity_metric'] = top20_summary['severity_metric'].round(6)

# 4) Up to 5 sample anomalous rows (compact)
# pick earliest anomalies across groups to show variety
sample_anomalies = (
    anomalies_all
    .sort_values(['ts', 'cmdb_id', 'kpi_name'])
    .loc[:, ['ts','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value']]
    .head(5)
    .copy()
)
# rename ts -> timestamp for output clarity
sample_anomalies = sample_anomalies.rename(columns={'ts':'timestamp'})

# Keep full metrics_agg and anomalies_all in kernel variables (already stored)
# Display compact outputs: top20 summary and up to 5 sample anomalies
top20_summary, sample_anomalies

```
Out[2]:
```
```python
# Build a plain-English summary string based on previously computed results (top20_summary, sample_anomalies)
try:
    top20_summary
    sample_anomalies
except NameError:
    raise RuntimeError("Required variables top20_summary and sample_anomalies not found in kernel.")

lines = []
lines.append("Summary of metric anomalies within 2020-05-25 04:30:00+08 through 2020-05-25 05:00:00+08:")
lines.append("")
# Top 5 from the top20_summary for concise reporting
lines.append("Top anomalous component-KPIs (top 5 of the top 20 by anomaly count):")
for r in top20_summary.reset_index(drop=True).head(5).itertuples(index=False):
    sev = r.severity_metric
    sev_str = "NaN" if pd.isna(sev) else f"{sev:.6f}"
    lines.append(f"- {r.cmdb_id} | {r.kpi_name}: {r.anomaly_count} anomalous points, earliest={r.earliest_anomaly_timestamp}, latest={r.latest_anomaly_timestamp}, severity_metric={sev_str}")

lines.append("")
lines.append("Notes on severity and thresholds:")
lines.append("- severity_metric is max relative deviation from the used threshold (p95 for high, p05 for low).")
lines.append("- 'NaN' severity means threshold was zero (division not possible) or no meaningful deviation could be computed.")
lines.append("")

lines.append("Up to 5 example anomalous points (timestamp, cmdb_id, kpi_name, value, flagged_side, used_threshold, threshold_value):")
for r in sample_anomalies.itertuples(index=False):
    lines.append(f"- {r.timestamp} | {r.cmdb_id} | {r.kpi_name} | value={r.value} | side={r.flagged_side} | used={r.used_threshold}({r.threshold_value})")

lines.append("")
lines.append("Data availability:")
lines.append("- Full per-(cmdb_id,kpi_name) thresholds table is in variable `metrics_agg`.")
lines.append("- Full anomaly rows detected within the window are in variable `anomalies_all`.")
lines.append("- The compact summary shown above is from `top20_summary` and sample rows from `sample_anomalies`.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id               kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_anomalous_value  min_anomalous_value  severity_metric
942   os_007            Disk_rd_ios             31  2020-05-25 04:30:00+08:00 2020-05-25 05:00:00+08:00             0.000000             0.000000              NaN
1587  os_020           Disk_io_util             31  2020-05-25 04:30:00+08:00 2020-05-25 05:00:00+08:00             0.005587             0.000000         0.007513
881   os_006             Agent_ping             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             1.000000             1.000000         0.000000
895   os_006            Disk_wr_ios             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.000000             0.000000              NaN
393   db_010                    AWS             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.020000             0.000000         1.000000
406   db_010              MEM_Total             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00           509.343945           509.343945         0.000000
405   db_010          Login_Per_Sec             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.200000             0.100000         0.000000
401   db_010                   Hang             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             1.000000             1.000000         0.000000
414   db_010  Physical_Read_Per_Sec             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00           117.600000             0.000000         0.000000
411   db_010           On_Off_State             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             1.000000             1.000000         0.000000
409   db_010          MEM_real_util             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00            57.790000            57.780000         0.000173
764   os_003                Page_pi             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.000000             0.000000              NaN
782   os_004             Agent_ping             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             1.000000             1.000000         0.000000
794   os_004            Disk_rd_kbs             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.000000             0.000000              NaN
793   os_004            Disk_rd_ios             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.000000             0.000000              NaN
992   os_008           Disk_io_util             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.000000             0.000000              NaN
391   db_010                    ACS             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.020000             0.010000         1.000000
390   db_009    tnsping_result_time             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00            10.000000             0.000000         0.000000
1003  os_008              ICMP_ping             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             1.000000             1.000000         0.000000
1016  os_008  Processor_load_15_min             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00             0.008333             0.008333         0.000000,                   timestamp cmdb_id             kpi_name  value flagged_side used_threshold  threshold_value
0 2020-05-25 04:30:00+08:00  db_001                  ACS   0.02         high            p95             0.02
1 2020-05-25 04:30:00+08:00  db_001                 AIOS   0.00          low            p05             0.00
2 2020-05-25 04:30:00+08:00  db_001                  AWS   0.00          low            p05             0.00
6 2020-05-25 04:30:00+08:00  db_001  DFParaWrite_Per_Sec   0.00          low            p05             0.00
9 2020-05-25 04:30:00+08:00  db_001                 Hang   1.00         high            p95             1.00)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse anomalies_all if present, otherwise recompute using metrics_agg and df_metric
try:
    anomalies_all
except NameError:
    # Ensure df_metric and metrics_agg exist
    try:
        df_metric
    except NameError:
        df_metric = pd.read_csv("metric.csv")
    try:
        metrics_agg
    except NameError:
        # compute global thresholds if missing
        metrics_agg = (
            df_metric
            .groupby(['cmdb_id', 'kpi_name'], as_index=False)
            .agg(
                count_of_points=('value', 'size'),
                min_timestamp=('timestamp', 'min'),
                max_timestamp=('timestamp', 'max'),
                global_p95=('value', lambda x: float(x.quantile(0.95))),
                global_p05=('value', lambda x: float(x.quantile(0.05)))
            )
        )
        metrics_agg['min_timestamp'] = pd.to_datetime(metrics_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        metrics_agg['max_timestamp'] = pd.to_datetime(metrics_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # prepare metric timestamps
    df_metric_ts = df_metric.copy()
    df_metric_ts['ts'] = pd.to_datetime(df_metric_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # window
    window_start = pd.Timestamp('2020-05-25 04:30:00', tz=tz)
    window_end = pd.Timestamp('2020-05-25 05:00:00', tz=tz)
    df_window = df_metric_ts[(df_metric_ts['ts'] >= window_start) & (df_metric_ts['ts'] <= window_end)].copy()
    # merge thresholds
    df_win_merged = df_window.merge(metrics_agg[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')
    # identify anomalies
    cond_high = df_win_merged['value'] >= df_win_merged['global_p95']
    cond_low = df_win_merged['value'] <= df_win_merged['global_p05']
    df_win_merged['flagged_side'] = np.where(cond_high, 'high', np.where(cond_low, 'low', None))
    anomalies_all = df_win_merged[df_win_merged['flagged_side'].notnull()].copy()
    anomalies_all['used_threshold'] = np.where(anomalies_all['flagged_side']=='high', 'p95', 'p05')
    anomalies_all['threshold_value'] = np.where(anomalies_all['flagged_side']=='high', anomalies_all['global_p95'], anomalies_all['global_p05'])
    # compute severity per row (abs relative)
    def compute_sev(r):
        th = r['threshold_value']
        if th == 0 or pd.isna(th):
            return np.nan
        return abs((r['value'] - th) / th)
    anomalies_all['severity_row'] = anomalies_all.apply(compute_sev, axis=1)

# Now build fault runs by grouping consecutive timestamps within 1 minute
df = anomalies_all.copy()

# Ensure ts is timezone-aware datetime
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)
else:
    # ensure tz convert
    df['ts'] = pd.to_datetime(df['ts']).dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(tz) if df['ts'].dtype == object else df['ts']

# Sort for grouping
df = df.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)

# Compute time difference in seconds within each (cmdb_id,kpi_name)
df['prev_ts'] = df.groupby(['cmdb_id','kpi_name'])['ts'].shift(1)
df['gap_secs'] = (df['ts'] - df['prev_ts']).dt.total_seconds().fillna(0)
# Start new run when gap > 60 seconds
df['new_run'] = (df['gap_secs'] > 60).astype(int)
# cumulative run id per group
df['run_seq'] = df.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()

# Build run_id combining keys
df['run_id'] = df['cmdb_id'].astype(str) + '||' + df['kpi_name'].astype(str) + '||' + df['run_seq'].astype(str)

# Aggregate runs
agg_funcs = {
    'ts': ['min','max','size'],
    'value': ['max','min'],
    'flagged_side': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0],
    'threshold_value': lambda x: x.loc[x.index[0]] if len(x.unique())==1 else x.iloc[0],
    'severity_row': 'max'
}

runs = df.groupby(['cmdb_id','kpi_name','run_seq','run_id'], as_index=False).agg({
    'ts': ['min','max','size'],
    'value': ['max','min'],
    'flagged_side': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0],
    'threshold_value': lambda x: x.iloc[0],
    'severity_row': 'max'
})

# Flatten columns
runs.columns = ['cmdb_id','kpi_name','run_seq','run_id',
                'fault_start_timestamp','fault_end_timestamp','num_points_in_run',
                'max_value','min_value','breach_side','used_threshold_value','run_severity']

# Convert timestamps to tz-aware (they already are)
runs['fault_start_timestamp'] = pd.to_datetime(runs['fault_start_timestamp']).dt.tz_convert(tz)
runs['fault_end_timestamp'] = pd.to_datetime(runs['fault_end_timestamp']).dt.tz_convert(tz)

# Handle run_severity NaN where used_threshold_value == 0 -> ensure NaN
runs['run_severity'] = runs['run_severity'].replace([np.inf, -np.inf], np.nan)
runs.loc[runs['used_threshold_value'] == 0, 'run_severity'] = np.nan

# 2) Filter out single-point runs unless run_severity > 0.5
runs_filtered = runs[(runs['num_points_in_run'] > 1) | (runs['run_severity'] > 0.5)].copy()

# 3A) faults_table: top 20 by num_points_in_run desc then run_severity desc
faults_table = runs_filtered.copy()
faults_table['run_severity'] = faults_table['run_severity'].astype(float)
faults_table = faults_table.sort_values(['num_points_in_run','run_severity'], ascending=[False, False]).reset_index(drop=True)
faults_table_top20 = faults_table.head(20).copy()
# Round severity for compactness
faults_table_top20['run_severity'] = faults_table_top20['run_severity'].round(6)

# Reorder columns as requested
faults_table_top20 = faults_table_top20[[
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'num_points_in_run','max_value','min_value','breach_side','used_threshold_value','run_severity'
]]

# 3B) component_summary aggregated per cmdb_id
component_summary = (
    runs_filtered
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_fault_runs = ('run_id','nunique'),
        total_fault_points = ('num_points_in_run','sum'),
        earliest_fault_start = ('fault_start_timestamp','min'),
        latest_fault_end = ('fault_end_timestamp','max'),
        max_run_severity = ('run_severity','max')
    )
)

component_summary['max_run_severity'] = component_summary['max_run_severity'].astype(float).round(6)
component_summary_top20 = component_summary.sort_values('total_fault_points', ascending=False).head(20).reset_index(drop=True)

# Keep full tables in kernel variables for later steps
faults_table_full = faults_table.copy()
component_summary_full = component_summary.copy()

# Final compact outputs (top 20 fault runs and top 20 components)
faults_table_top20, component_summary_top20

```
Out[3]:
```
Summary of fault-run extraction (2020-05-25 04:30:00+08 through 05:00:00+08)

What was done
- Consecutive anomalous metric points (within 1 minute gaps) were merged into fault runs per (cmdb_id, kpi_name).
- Single-point runs were removed unless their run_severity > 0.5.
- Outputs: (A) faults_table (top 20 fault runs by run length then severity) and (B) component_summary (top 20 components by total_fault_points).

Top fault runs (top 5 of the faults_table)
1. os_007 | Disk_rd_ios — 31 points, 2020-05-25 04:30 → 05:00, breach_side=high, used_threshold=0.00, run_severity=NaN (threshold=0 => severity undefined)
2. os_020 | Disk_io_util — 31 points, 2020-05-25 04:30 → 05:00, breach_side=low, used_threshold=0.00, run_severity=NaN
3. db_002 | Login_Per_Sec — 30 points, 04:30 → 04:59, max=2.40 min=0.40, breach_side=high, threshold=0.40, run_severity=5.000000 (very large relative breach)
4. db_010 | ACS — 30 points, 04:30 → 04:59, breach_side=high, threshold=0.01, run_severity=1.000000
5. db_011 | ACS — 30 points, 04:30 → 04:59, breach_side=low, threshold=0.01, run_severity=1.000000

Other notable runs (examples)
- redis_012 | instantaneous_input_kbps — 30 points, run_severity=0.625
- os_007 | Disk_svctm — 30 points, run_severity=0.700000
- Several tnsping_result_time runs (db_011, db_013) show 30-point high breaches with run_severity=0.5

Component-level summary (top components by total_fault_points)
- db_006: 48 runs, 597 fault points, earliest start 04:30, latest end 04:59, max_run_severity=0.451381
- redis_012: 27 runs, 543 points, max_run_severity=0.625000
- db_005: 35 runs, 526 points, max_run_severity=0.886305
- redis_004: 33 runs, 523 points, max_run_severity=1.000000
- db_002: 43 runs, 484 points, max_run_severity=59.922730 (extremely large relative breach — likely due to very small threshold)

Notes and caveats
- run_severity = max(|value - threshold| / threshold) across the run. If threshold == 0, run_severity is NaN (cannot compute relative ratio) — several disk/network metrics fell into this category.
- Extremely large severity values indicate very small threshold values; interpret absolute-value context as well (check actual threshold and values).
- The full results are stored in kernel variables faults_table_full (all fault runs) and component_summary_full (all per-component aggregates) for further trace/log analysis.

If you want, I can:
- show the full top-20 faults_table and component_summary as plain tables, or
- inspect specific high-severity runs (e.g., db_002 or db_001) and list their raw anomalous points for troubleshooting.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                   kpi_name     fault_start_timestamp       fault_end_timestamp  num_points_in_run  max_value  min_value breach_side  used_threshold_value  run_severity
0      os_007                Disk_rd_ios 2020-05-25 04:30:00+08:00 2020-05-25 05:00:00+08:00                 31   0.000000       0.00        high                  0.00           NaN
1      os_020               Disk_io_util 2020-05-25 04:30:00+08:00 2020-05-25 05:00:00+08:00                 31   0.005587       0.00         low                  0.00           NaN
2      db_002              Login_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   2.400000       0.40        high                  0.40      5.000000
3      db_010                        ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.020000       0.01        high                  0.01      1.000000
4      db_011                        ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.020000       0.00         low                  0.01      1.000000
5      os_007                 Disk_svctm 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   1.700000       0.00         low                  1.00      0.700000
6   redis_012   instantaneous_input_kbps 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.090000       0.03        high                  0.08      0.625000
7      db_004                        ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.020000       0.01        high                  0.02      0.500000
8      db_011        tnsping_result_time 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30  30.000000      10.00        high                 20.00      0.500000
9      db_013        tnsping_result_time 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30  30.000000      10.00        high                 20.00      0.500000
10     os_006                    Page_po 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   3.000000       1.00         low                  2.00      0.500000
11     db_005                        ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.030000       0.01        high                  0.02      0.500000
12     db_007                        ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.030000       0.01         low                  0.02      0.500000
13     db_013              Login_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.300000       0.10         low                  0.10      0.500000
14     db_004              Login_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   1.700000       1.20         low                  1.20      0.307692
15     db_008                Sess_Active 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   0.500000       0.10         low                  0.10      0.250000
16     os_013                Disk_wr_ios 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   6.000000       5.00        high                  5.00      0.200000
17  redis_012  instantaneous_ops_per_sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30  11.000000       9.00        high                 11.00      0.100000
18     db_007              Proc_Used_Pct 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   1.940000       1.92        high                  1.93      0.005181
19     db_007         Proc_User_Used_Pct 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   1.940000       1.92        high                  1.93      0.005181,       cmdb_id  total_fault_runs  total_fault_points      earliest_fault_start          latest_fault_end  max_run_severity
0      db_006                48                 597 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.451381
1   redis_012                27                 543 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.625000
2      db_005                35                 526 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.886305
3   redis_004                33                 523 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.000000
4   redis_008                26                 514 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.500000
5      db_013                26                 486 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.500000
6      db_002                43                 484 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00         59.922730
7      db_008                28                 464 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.666667
8      os_006                33                 463 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.679142
9      os_007                35                 461 2020-05-25 04:30:00+08:00 2020-05-25 05:00:00+08:00          1.000000
10     db_010                27                 450 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.000000
11     db_011                35                 447 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          3.009022
12     os_008                38                 434 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.000000
13     db_007                41                 425 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00         38.799398
14     db_001                45                 418 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00         65.645067
15     db_012                28                 414 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00         66.394541
16  redis_010                19                 401 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.720377
17     os_014                34                 398 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.404707
18  redis_006                24                 392 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          0.731826
19     os_016                32                 384 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00          1.222222)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist in kernel: faults_table_full and anomalies_all
try:
    faults_table_full
    anomalies_all
except NameError as e:
    raise RuntimeError("Required kernel variables missing: faults_table_full and/or anomalies_all") from e

# DB candidates
db_candidates = ['db_012','db_001','db_002','db_007']

# Work on a copy of faults_table_full to avoid mutating kernel state
ft = faults_table_full.copy()

# Ensure run_severity is numeric
ft['run_severity'] = pd.to_numeric(ft['run_severity'], errors='coerce')

# Filter to DB candidates
ft_db = ft[ft['cmdb_id'].isin(db_candidates)].copy()

# Sort by run_severity desc (NaN last) then num_points_in_run desc
ft_db_sorted = ft_db.sort_values(['run_severity','num_points_in_run'], ascending=[False, False], na_position='last').reset_index(drop=True)

# Prepare faults_detail for display: include requested columns and limit to top 20
display_cols = ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                'num_points_in_run','max_value','min_value','breach_side','used_threshold_value','run_severity','run_id']
# Some rows might lack run_id if earlier pipeline differed, guard for that
for c in display_cols:
    if c not in ft_db_sorted.columns:
        ft_db_sorted[c] = np.nan

faults_detail = ft_db_sorted[display_cols].head(20).copy()

# For compactness, round run_severity
faults_detail['run_severity'] = faults_detail['run_severity'].round(6)

# Now prepare severe_run_samples:
# Recompute run_id on anomalies_all to ensure matching scheme (cmdb_id||kpi_name||run_seq)
an = anomalies_all.copy()

# Ensure timestamp column 'ts' exists and is timezone-aware
if 'ts' in an.columns:
    an['ts'] = pd.to_datetime(an['ts'])
else:
    # try 'timestamp' column
    an['ts'] = pd.to_datetime(an['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Sort and compute gaps to assign run_seq
an = an.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)
an['prev_ts'] = an.groupby(['cmdb_id','kpi_name'])['ts'].shift(1)
an['gap_secs'] = (an['ts'] - an['prev_ts']).dt.total_seconds().fillna(0)
an['new_run'] = (an['gap_secs'] > 60).astype(int)
an['run_seq'] = an.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum().astype(int)
an['run_id'] = an['cmdb_id'].astype(str) + '||' + an['kpi_name'].astype(str) + '||' + an['run_seq'].astype(str)

# Ensure used_threshold and threshold_value exist (create if missing)
if 'used_threshold' not in an.columns or 'threshold_value' not in an.columns:
    an['used_threshold'] = np.where(an['flagged_side']=='high', 'p95', 'p05')
    an['threshold_value'] = np.where(an['flagged_side']=='high', an.get('global_p95', np.nan), an.get('global_p05', np.nan))

# From faults_detail pick the run with highest run_severity (and tie-break by num_points_in_run)
# Use ft_db_sorted (which includes run_id) to select the top run among DBs
if ft_db_sorted.shape[0] == 0:
    # no runs found for these DBs
    severe_run_samples = pd.DataFrame(columns=['ts','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value'])
else:
    top_run_row = ft_db_sorted.iloc[0]
    chosen_run_id = top_run_row.get('run_id', None)
    # If run_id is missing, fall back to matching by keys and timestamps
    if pd.isna(chosen_run_id) or chosen_run_id is None:
        # find by cmdb_id, kpi_name, and start/end timestamps
        mask = (
            (an['cmdb_id'] == top_run_row['cmdb_id']) &
            (an['kpi_name'] == top_run_row['kpi_name']) &
            (an['ts'] >= pd.to_datetime(top_run_row['fault_start_timestamp'])) &
            (an['ts'] <= pd.to_datetime(top_run_row['fault_end_timestamp']))
        )
        samples = an[mask].sort_values('ts').head(10)
    else:
        samples = an[an['run_id'] == chosen_run_id].sort_values('ts').head(10)
    # Select requested columns and rename ts->timestamp for output clarity
    if not samples.empty:
        severe_run_samples = samples.loc[:, ['ts','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value']].copy()
        severe_run_samples = severe_run_samples.rename(columns={'ts':'timestamp'})
    else:
        severe_run_samples = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value'])

# Final outputs: faults_detail (top up to 20 runs for DBs) and severe_run_samples (up to 10 rows for most severe run)
# Keep full results in kernel variables for later use
faults_detail_db = faults_detail.copy()
severe_run_samples_db = severe_run_samples.copy()

# Display compact results (variables)
faults_detail_db, severe_run_samples_db

```
Out[4]:
```
Faults_detail (filtered to db_012, db_001, db_002, db_007) — top runs sorted by run_severity desc then run length:
Columns: cmdb_id | kpi_name | fault_start → fault_end | pts | max_value | min_value | breach_side | used_threshold_value | run_severity

1) db_012 | SeqRead_Per_Sec | 2020-05-25 04:31 → 04:31 | 1 | 13.580 | 13.580 | high | 0.20150 | 66.394541
2) db_001 | LFParaWrite_Per_Sec | 2020-05-25 04:54 → 04:54 | 1 | 205.700 | 205.700 | high | 3.08650 | 65.645067
3) db_002 | Logic_Read_Per_Sec | 2020-05-25 04:47 → 04:47 | 1 | 16730.600 | 16730.600 | high | 274.62000 | 59.922730
4) db_007 | DFParaWrite_Per_Sec | 2020-05-25 04:53 → 04:53 | 1 | 79.360 | 79.360 | high | 1.99400 | 38.799398
5) db_001 | DFParaWrite_Per_Sec | 2020-05-25 04:52 → 04:53 | 2 | 51.020 | 0.000 | high | 2.72900 | 17.695493
6) db_001 | LFParaWrite_Per_Sec | 2020-05-25 04:56 → 04:56 | 1 | 22.290 | 22.290 | high | 3.08650 | 6.221772
7) db_007 | DFParaWrite_Per_Sec | 2020-05-25 04:51 → 04:51 | 1 | 13.090 | 13.090 | high | 1.99400 | 5.564694
8) db_002 | Login_Per_Sec | 2020-05-25 04:30 → 04:59 | 30 | 2.400 | 0.400 | high | 0.40000 | 5.000000
9) db_001 | Login_Per_Sec | 2020-05-25 04:36 → 04:36 | 1 | 1.500 | 1.500 | high | 0.30000 | 4.000000
10) db_001 | Logic_Read_Per_Sec | 2020-05-25 04:36 → 04:36 | 1 | 9415.700 | 9415.700 | high | 2321.30000 | 3.056218
11) db_007 | Session_pct | 2020-05-25 04:58 → 04:58 | 1 | 0.084 | 0.084 | high | 0.02515 | 2.339960
12) db_007 | DbTime | 2020-05-25 04:58 → 04:58 | 1 | 29.600 | 29.600 | high | 9.05500 | 2.268912
13) db_012 | DFParaWrite_Per_Sec | 2020-05-25 04:51 → 04:54 | 4 | 0.260 | 0.050 | low | 0.05000 | 2.250000
14) db_007 | LFSync_Per_Sec | 2020-05-25 04:55 → 04:56 | 2 | 39.130 | 23.550 | high | 14.85300 | 1.634485
15) db_007 | LFParaWrite_Per_Sec | 2020-05-25 04:55 → 04:56 | 2 | 14.530 | 8.920 | high | 5.80150 | 1.504525
16) db_001 | ACS | 2020-05-25 04:48 → 04:48 | 1 | 0.000 | 0.000 | low | 0.00950 | 1.000000
17) db_001 | ACS | 2020-05-25 04:58 → 04:58 | 1 | 0.000 | 0.000 | low | 0.00950 | 1.000000
18) db_002 | Exec_Per_Sec | 2020-05-25 04:58 → 04:58 | 1 | 45.100 | 45.100 | high | 23.78500 | 0.896153
19) db_007 | Logic_Read_Per_Sec | 2020-05-25 04:31 → 04:31 | 1 | 4402.200 | 4402.200 | high | 2515.69000 | 0.749898
20) db_002 | Logic_Read_Per_Sec | 2020-05-25 04:58 → 04:58 | 1 | 480.200 | 480.200 | high | 274.62000 | 0.748598

Severe_run_samples — raw anomalous points for the single most severe run among these DBs:
(The highest run_severity run is db_012 | SeqRead_Per_Sec with run_severity 66.394541)

Columns: timestamp | cmdb_id | kpi_name | value | flagged_side | used_threshold | threshold_value

- 2020-05-25 04:31:00+08:00 | db_012 | SeqRead_Per_Sec | 13.58 | high | p95 | 0.2015

Notes / quick interpretation
- Extremely large run_severity values (e.g., >50) indicate the threshold is very small relative to the observed value — check absolute values and thresholds before concluding impact.
- Many runs are single-point high breaches (several LFParaWrite, Logic_Read, etc.) — these are flagged because they greatly exceed small thresholds.
- There is one long run for db_002 | Login_Per_Sec (30 points) with significant severity (5.0) and multiple multi-point runs for db_012 and db_007 of interest.
- Full details are stored in kernel variables faults_table_full and anomalies_all for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id             kpi_name     fault_start_timestamp       fault_end_timestamp  num_points_in_run  max_value  min_value breach_side  used_threshold_value  run_severity                          run_id
0   db_012      SeqRead_Per_Sec 2020-05-25 04:31:00+08:00 2020-05-25 04:31:00+08:00                  1     13.580     13.580        high               0.20150     66.394541      db_012||SeqRead_Per_Sec||0
1   db_001  LFParaWrite_Per_Sec 2020-05-25 04:54:00+08:00 2020-05-25 04:54:00+08:00                  1    205.700    205.700        high               3.08650     65.645067  db_001||LFParaWrite_Per_Sec||3
2   db_002   Logic_Read_Per_Sec 2020-05-25 04:47:00+08:00 2020-05-25 04:47:00+08:00                  1  16730.600  16730.600        high             274.62000     59.922730   db_002||Logic_Read_Per_Sec||1
3   db_007  DFParaWrite_Per_Sec 2020-05-25 04:53:00+08:00 2020-05-25 04:53:00+08:00                  1     79.360     79.360        high               1.99400     38.799398  db_007||DFParaWrite_Per_Sec||2
4   db_001  DFParaWrite_Per_Sec 2020-05-25 04:52:00+08:00 2020-05-25 04:53:00+08:00                  2     51.020      0.000        high               2.72900     17.695493  db_001||DFParaWrite_Per_Sec||6
5   db_001  LFParaWrite_Per_Sec 2020-05-25 04:56:00+08:00 2020-05-25 04:56:00+08:00                  1     22.290     22.290        high               3.08650      6.221772  db_001||LFParaWrite_Per_Sec||4
6   db_007  DFParaWrite_Per_Sec 2020-05-25 04:51:00+08:00 2020-05-25 04:51:00+08:00                  1     13.090     13.090        high               1.99400      5.564694  db_007||DFParaWrite_Per_Sec||1
7   db_002        Login_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30      2.400      0.400        high               0.40000      5.000000        db_002||Login_Per_Sec||0
8   db_001        Login_Per_Sec 2020-05-25 04:36:00+08:00 2020-05-25 04:36:00+08:00                  1      1.500      1.500        high               0.30000      4.000000        db_001||Login_Per_Sec||0
9   db_001   Logic_Read_Per_Sec 2020-05-25 04:36:00+08:00 2020-05-25 04:36:00+08:00                  1   9415.700   9415.700        high            2321.30000      3.056218   db_001||Logic_Read_Per_Sec||1
10  db_007          Session_pct 2020-05-25 04:58:00+08:00 2020-05-25 04:58:00+08:00                  1      0.084      0.084        high               0.02515      2.339960          db_007||Session_pct||3
11  db_007               DbTime 2020-05-25 04:58:00+08:00 2020-05-25 04:58:00+08:00                  1     29.600     29.600        high               9.05500      2.268912               db_007||DbTime||1
12  db_012  DFParaWrite_Per_Sec 2020-05-25 04:51:00+08:00 2020-05-25 04:54:00+08:00                  4      0.260      0.050         low               0.05000      2.250000  db_012||DFParaWrite_Per_Sec||5
13  db_007       LFSync_Per_Sec 2020-05-25 04:55:00+08:00 2020-05-25 04:56:00+08:00                  2     39.130     23.550        high              14.85300      1.634485       db_007||LFSync_Per_Sec||1
14  db_007  LFParaWrite_Per_Sec 2020-05-25 04:55:00+08:00 2020-05-25 04:56:00+08:00                  2     14.530      8.920        high               5.80150      1.504525  db_007||LFParaWrite_Per_Sec||1
15  db_001                  ACS 2020-05-25 04:48:00+08:00 2020-05-25 04:48:00+08:00                  1      0.000      0.000         low               0.00950      1.000000                  db_001||ACS||5
16  db_001                  ACS 2020-05-25 04:58:00+08:00 2020-05-25 04:58:00+08:00                  1      0.000      0.000         low               0.00950      1.000000                  db_001||ACS||9
17  db_002         Exec_Per_Sec 2020-05-25 04:58:00+08:00 2020-05-25 04:58:00+08:00                  1     45.100     45.100        high              23.78500      0.896153         db_002||Exec_Per_Sec||8
18  db_007   Logic_Read_Per_Sec 2020-05-25 04:31:00+08:00 2020-05-25 04:31:00+08:00                  1   4402.200   4402.200        high            2515.69000      0.749898   db_007||Logic_Read_Per_Sec||0
19  db_002   Logic_Read_Per_Sec 2020-05-25 04:58:00+08:00 2020-05-25 04:58:00+08:00                  1    480.200    480.200        high             274.62000      0.748598   db_002||Logic_Read_Per_Sec||3,                      timestamp cmdb_id         kpi_name  value flagged_side used_threshold  threshold_value
6864 2020-05-25 04:31:00+08:00  db_012  SeqRead_Per_Sec  13.58         high            p95           0.2015)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing df_trace if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp column exists and compute timezone-aware ts column
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global_p95 per (cmdb_id, trace_name) using full series (no filtering)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_of_points=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        global_p95=('value', lambda x: float(x.quantile(0.95)))
    )
)

# convert min/max timestamps to tz-aware datetimes for clarity
trace_thresholds['min_timestamp'] = pd.to_datetime(trace_thresholds['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_thresholds['max_timestamp'] = pd.to_datetime(trace_thresholds['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Filter trace points to the incident window (inclusive)
window_start = pd.Timestamp('2020-05-25 04:30:00', tz=tz)
window_end = pd.Timestamp('2020-05-25 05:00:00', tz=tz)

df_trace_window = df_trace[(df_trace['ts'] >= window_start) & (df_trace['ts'] <= window_end)].copy()

# 3) Flag anomalies where value >= global_p95
df_trace_window = df_trace_window.merge(
    trace_thresholds[['cmdb_id','trace_name','global_p95']],
    on=['cmdb_id','trace_name'],
    how='left'
)
df_trace_window['is_anomaly'] = df_trace_window['value'] >= df_trace_window['global_p95']

# Store full anomalies for later use
trace_anomalies_all = df_trace_window[df_trace_window['is_anomaly']].copy()

# 4) Compact summary: top 20 groups by anomaly_count
traces_summary = (
    trace_anomalies_all
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        anomaly_count=('value','size'),
        earliest_anomaly_timestamp=('ts','min'),
        latest_anomaly_timestamp=('ts','max'),
        global_p95=('global_p95','first')
    )
)

traces_summary_top20 = traces_summary.sort_values('anomaly_count', ascending=False).head(20).copy()
traces_summary_top20['global_p95'] = traces_summary_top20['global_p95'].round(6)

# 5) Up to 10 sample anomalous trace rows restricted to given DB candidates
db_candidates = ['db_002','db_001','db_007','db_012']
sample_trace_anomalies = (
    trace_anomalies_all[trace_anomalies_all['cmdb_id'].isin(db_candidates)]
    .sort_values('ts')
    .loc[:, ['ts','cmdb_id','trace_name','value','global_p95']]
    .rename(columns={'ts':'timestamp'})
    .head(10)
    .copy()
)
sample_trace_anomalies['global_p95'] = sample_trace_anomalies['global_p95'].round(6)

# Keep full thresholds and anomalies in kernel variables (trace_thresholds, trace_anomalies_all)
# Final compact outputs
traces_summary_top20, sample_trace_anomalies

```
Out[5]:
```
```python
# Plain-English summary string based on last trace analysis results (variables: traces_summary_top20, sample_trace_anomalies, trace_thresholds, trace_anomalies_all)
summary_lines = []
summary_lines.append("Trace anomaly summary (2020-05-25 04:30+08 through 05:00+08):")
summary_lines.append("")
summary_lines.append("1) Top groups by anomaly count (top 20):")
summary_lines.append("- Most top groups are docker/os error_rate traces (e.g., trace.self.csf.error_rate, trace.from_...remoteprocess.error_rate).")
summary_lines.append("- Many of these groups have anomaly_count = 30 (anomalous at every minute in the window).")
summary_lines.append("- The computed global_p95 for these top groups is 0.000, so points >= 0.0 were flagged (check whether this is expected for error_rate metrics).")
summary_lines.append("")
summary_lines.append("2) DB-focused sample rows:")
summary_lines.append("- No anomalous trace rows were found in the window for DB candidates ['db_002','db_001','db_007','db_012']; sample set is empty.")
summary_lines.append("")
summary_lines.append("3) Notes and next steps:")
summary_lines.append("- global_p95 was computed on the full trace series before window filtering (as requested).")
summary_lines.append("- Because many global_p95 values are zero, interpret anomalies carefully — a nonzero value may trivially exceed a zero p95.")
summary_lines.append("- Full per-(cmdb_id,trace_name) thresholds are in variable `trace_thresholds` and all flagged trace points in the window are in `trace_anomalies_all` for deeper inspection.")
summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                    trace_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  global_p95
2    docker_001    trace.from_os_022.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
6    docker_001                     trace.self.csf.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
14   docker_001                    trace.self.jdbc.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
10   docker_001               trace.self.flyremote.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
78   docker_003  trace.to_docker_005.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
54   docker_002  trace.to_docker_008.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
58   docker_003    trace.from_os_021.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
66   docker_003               trace.self.flyremote.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
34   docker_002                     trace.self.csf.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
38   docker_002               trace.self.flyremote.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
46   docker_002                   trace.self.local.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
42   docker_002                    trace.self.jdbc.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
26   docker_001  trace.to_docker_008.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
30   docker_002    trace.from_os_022.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
22   docker_001  trace.to_docker_007.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
18   docker_001                   trace.self.local.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
50   docker_002  trace.to_docker_007.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
198      os_022                     trace.self.csf.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
206      os_022  trace.to_docker_002.remoteprocess.error_rate             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.000
153  docker_007                  trace.self.jdbc.duration_p95             30  2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00       0.008, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, global_p95]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required kernel variables exist
try:
    faults_table_full
    anomalies_all
except NameError as e:
    raise RuntimeError("Required kernel variables faults_table_full and anomalies_all are missing.") from e

# 1) db002_fault_runs: filter faults_table_full for cmdb_id='db_002'
ft = faults_table_full.copy()

# Ensure run_severity numeric
ft['run_severity'] = pd.to_numeric(ft.get('run_severity'), errors='coerce')

# Filter to db_002
db002_runs = ft[ft['cmdb_id'] == 'db_002'].copy()

# Sort by num_points_in_run desc then run_severity desc (NaNs last)
db002_runs_sorted = db002_runs.sort_values(
    by=['num_points_in_run','run_severity'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Ensure requested columns exist; add if missing with NaN
requested_cols = [
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'num_points_in_run','max_value','min_value','breach_side','used_threshold_value','run_severity'
]
for c in requested_cols:
    if c not in db002_runs_sorted.columns:
        db002_runs_sorted[c] = np.nan

# Prepare final faults table (top 20) and round severity for compactness
db002_fault_runs = db002_runs_sorted[requested_cols].head(20).copy()
db002_fault_runs['run_severity'] = pd.to_numeric(db002_fault_runs['run_severity'], errors='coerce').round(6)

# 2) db002_login_anomalies: all raw anomalous metric points for db_002 & Login_Per_Sec within window
an = anomalies_all.copy()

# Ensure timestamp column 'ts' or 'timestamp' present and timezone-aware
if 'ts' in an.columns:
    an['ts'] = pd.to_datetime(an['ts'])
else:
    # try to construct from 'timestamp' which may already be tz-aware or epoch seconds
    if 'timestamp' in an.columns:
        # if timestamp dtype is integer/float, convert from epoch; else parse
        if np.issubdtype(an['timestamp'].dtype, np.number):
            an['ts'] = pd.to_datetime(an['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            an['ts'] = pd.to_datetime(an['timestamp']).dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        raise RuntimeError("anomalies_all lacks both 'ts' and 'timestamp' columns.")

# Ensure ts is tz-aware in Asia/Shanghai
if an['ts'].dt.tz is None:
    an['ts'] = an['ts'].dt.tz_localize('UTC').dt.tz_convert(tz)
else:
    an['ts'] = an['ts'].dt.tz_convert(tz)

# Define window (inclusive)
window_start = pd.Timestamp('2020-05-25 04:30:00', tz=tz)
window_end = pd.Timestamp('2020-05-25 05:00:00', tz=tz)

# Ensure used_threshold and threshold_value exist; if not, attempt to derive from metrics_agg
if 'used_threshold' not in an.columns or 'threshold_value' not in an.columns:
    try:
        metrics_agg  # use existing thresholds table
    except NameError:
        # compute thresholds if missing (global p95/p05) from df_metric
        try:
            df_metric
        except NameError:
            df_metric = pd.read_csv("metric.csv")
        metrics_agg = (
            df_metric
            .groupby(['cmdb_id','kpi_name'], as_index=False)
            .agg(global_p95=('value', lambda x: float(x.quantile(0.95))),
                 global_p05=('value', lambda x: float(x.quantile(0.05))))
        )
    # merge thresholds into anomalies table
    an = an.merge(metrics_agg[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')
    an['used_threshold'] = np.where(an['flagged_side']=='high','p95','p05')
    an['threshold_value'] = np.where(an['flagged_side']=='high', an['global_p95'], an['global_p05'])

# Filter for db_002 and Login_Per_Sec within window
mask = (
    (an['cmdb_id'] == 'db_002') &
    (an['kpi_name'] == 'Login_Per_Sec') &
    (an['ts'] >= window_start) &
    (an['ts'] <= window_end)
)
db002_login = an[mask].copy()

# Select and rename columns as requested, sort ascending by timestamp, limit to 60 rows
cols_out = ['ts','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value']
for c in cols_out:
    if c not in db002_login.columns:
        db002_login[c] = np.nan

db002_login_anomalies = db002_login.loc[:, cols_out].rename(columns={'ts':'timestamp'}).sort_values('timestamp').head(60).reset_index(drop=True)

# Ensure timestamp column is tz-aware Asia/Shanghai
db002_login_anomalies['timestamp'] = pd.to_datetime(db002_login_anomalies['timestamp']).dt.tz_convert(tz)

# Final compact outputs (variables to be displayed)
db002_fault_runs, db002_login_anomalies

```
Out[6]:
```
```python
# Plain-English summary string based on kernel results (db002_fault_runs, db002_login_anomalies)
summary = (
    "db_002 summary (2020-05-25 04:30+08 through 05:00+08):\n"
    "- Top fault run: Login_Per_Sec — 30 anomalous points from 2020-05-25 04:30 to 04:59, max=2.40, min=0.40, flagged HIGH vs p95=0.40, run_severity=5.0.\n"
    "- Multiple other long runs (30 points) for db_002: ACS, Hang, MEM_Total, On_Off_State, Proc_Used_Pct, Proc_User_Used_Pct, tnsping_result_time, etc.; several of these have global thresholds equal to 0 causing run_severity to be NaN (relative severity not computable).\n"
    "- In total, Login_Per_Sec produced 30 raw anomalous metric points (one per minute) — all values = 0.4 (equal to the p95) and flagged as 'high'. These 30 raw rows are in variable `db002_login_anomalies`.\n"
    "- The faults table for db_002 (variable `db002_fault_runs`) lists up to 20 runs; the top entries show sustained anomalies for Login_Per_Sec and many other metrics across the incident window.\n"
    "- Notes: Extremely large or NaN severity values arise when thresholds are very small or zero; interpret those with absolute values and thresholds in mind."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id               kpi_name     fault_start_timestamp       fault_end_timestamp  num_points_in_run   max_value   min_value breach_side  used_threshold_value  run_severity
0   db_002          Login_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    2.400000    0.400000        high              0.400000      5.000000
1   db_002                    ACS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    0.020000    0.010000        high              0.020000      0.000000
2   db_002                   Hang 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    1.000000    1.000000        high              1.000000      0.000000
3   db_002              MEM_Total 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30  381.902264  381.902264        high            381.902264      0.000000
4   db_002           On_Off_State 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    1.000000    1.000000        high              1.000000      0.000000
5   db_002          Proc_Used_Pct 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    1.350000    1.340000         low              1.340000      0.000000
6   db_002     Proc_User_Used_Pct 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    1.350000    1.340000         low              1.340000      0.000000
7   db_002                   AIOS 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    0.000000    0.000000        high              0.000000           NaN
8   db_002               Row_Lock 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    0.000000    0.000000        high              0.000000           NaN
9   db_002        SctRead_Per_Sec 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30    0.000000    0.000000        high              0.000000           NaN
10  db_002    tnsping_result_time 2020-05-25 04:30:00+08:00 2020-05-25 04:59:00+08:00                 30   10.000000    0.000000         low              0.000000           NaN
11  db_002                    AWS 2020-05-25 04:30:00+08:00 2020-05-25 04:57:00+08:00                 28    0.020000    0.000000         low              0.000000           NaN
12  db_002  Physical_Read_Per_Sec 2020-05-25 04:49:00+08:00 2020-05-25 04:59:00+08:00                 11   16.600000    0.000000         low              0.000000           NaN
13  db_002            Sess_Active 2020-05-25 04:34:00+08:00 2020-05-25 04:42:00+08:00                  9    0.200000    0.000000        high              0.200000      0.000000
14  db_002            TPS_Per_Sec 2020-05-25 04:45:00+08:00 2020-05-25 04:53:00+08:00                  9    1.700000    1.500000         low              1.500000      0.000000
15  db_002           Sess_Connect 2020-05-25 04:41:00+08:00 2020-05-25 04:48:00+08:00                  8  281.000000  278.000000         low            279.000000      0.003584
16  db_002          MEM_real_util 2020-05-25 04:31:00+08:00 2020-05-25 04:37:00+08:00                  7   45.200000   45.200000         low             45.200000      0.000000
17  db_002  Physical_Read_Per_Sec 2020-05-25 04:39:00+08:00 2020-05-25 04:45:00+08:00                  7    0.000000    0.000000         low              0.000000           NaN
18  db_002            Sess_Active 2020-05-25 04:54:00+08:00 2020-05-25 04:59:00+08:00                  6    0.300000    0.000000        high              0.200000      0.500000
19  db_002          MEM_real_util 2020-05-25 04:39:00+08:00 2020-05-25 04:43:00+08:00                  5   45.200000   45.200000         low             45.200000      0.000000,                    timestamp cmdb_id       kpi_name  value flagged_side used_threshold  threshold_value
0  2020-05-25 04:30:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
1  2020-05-25 04:31:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
2  2020-05-25 04:32:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
3  2020-05-25 04:33:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
4  2020-05-25 04:34:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
..                       ...     ...            ...    ...          ...            ...              ...
25 2020-05-25 04:55:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
26 2020-05-25 04:56:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
27 2020-05-25 04:57:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
28 2020-05-25 04:58:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4
29 2020-05-25 04:59:00+08:00  db_002  Login_Per_Sec    0.4         high            p95              0.4

[30 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse anomalies_all if available, otherwise recompute from df_metric and metrics_agg
try:
    anomalies_all
except NameError:
    # attempt to reconstruct anomalies_all following previous logic
    try:
        df_metric
    except NameError:
        df_metric = pd.read_csv("metric.csv")
    try:
        metrics_agg
    except NameError:
        metrics_agg = (
            df_metric
            .groupby(['cmdb_id','kpi_name'], as_index=False)
            .agg(global_p95=('value', lambda x: float(x.quantile(0.95))),
                 global_p05=('value', lambda x: float(x.quantile(0.05))))
        )
    # prepare timestamps
    df_metric_ts = df_metric.copy()
    df_metric_ts['ts'] = pd.to_datetime(df_metric_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # window
    window_start = pd.Timestamp('2020-05-25 04:30:00', tz=tz)
    window_end = pd.Timestamp('2020-05-25 05:00:00', tz=tz)
    df_window = df_metric_ts[(df_metric_ts['ts'] >= window_start) & (df_metric_ts['ts'] <= window_end)].copy()
    # merge thresholds
    anomalies_all = df_window.merge(metrics_agg[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')
    anomalies_all['flagged_side'] = None
    anomalies_all.loc[anomalies_all['value'] >= anomalies_all['global_p95'], 'flagged_side'] = 'high'
    anomalies_all.loc[anomalies_all['value'] <= anomalies_all['global_p05'], 'flagged_side'] = 'low'
    anomalies_all = anomalies_all[anomalies_all['flagged_side'].notnull()].copy()
    anomalies_all['used_threshold'] = anomalies_all['flagged_side'].map({'high':'p95','low':'p05'})
    anomalies_all['threshold_value'] = anomalies_all.apply(lambda r: r['global_p95'] if r['flagged_side']=='high' else r['global_p05'], axis=1)

# Ensure ts column exists and is tz-aware
if 'ts' in anomalies_all.columns:
    anomalies_all['ts'] = pd.to_datetime(anomalies_all['ts']).dt.tz_convert(tz)
elif 'timestamp' in anomalies_all.columns:
    # if timestamp is epoch seconds, convert; else parse
    if pd.api.types.is_numeric_dtype(anomalies_all['timestamp']):
        anomalies_all['ts'] = pd.to_datetime(anomalies_all['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        anomalies_all['ts'] = pd.to_datetime(anomalies_all['timestamp']).dt.tz_localize('UTC').dt.tz_convert(tz)
else:
    raise RuntimeError("anomalies_all lacks timestamp column")

# Target timestamp (tz-aware)
target_ts = pd.Timestamp('2020-05-25 04:30:00', tz=tz)

# A) All anomalous metric rows for cmdb_id='db_002' at the target timestamp
mask_a = (
    (anomalies_all['cmdb_id'] == 'db_002') &
    (anomalies_all['ts'] == target_ts)
)
db002_0430_anomalies = anomalies_all.loc[mask_a, ['ts','cmdb_id','kpi_name','value','flagged_side','used_threshold','threshold_value']].copy()
# rename ts -> timestamp for output clarity
db002_0430_anomalies = db002_0430_anomalies.rename(columns={'ts':'timestamp'}).reset_index(drop=True)

# B) Earliest anomaly timestamp for each KPI in the requested list for db_002
kpis_of_interest = ['Hang','On_Off_State','tnsping_result_time','Login_Per_Sec','ACS','AWS','Sess_Active','DbTime']

# Filter anomalies_all for db_002 and those KPIs
mask_b = (anomalies_all['cmdb_id'] == 'db_002') & (anomalies_all['kpi_name'].isin(kpis_of_interest))
df_db002_kpis = anomalies_all.loc[mask_b, ['kpi_name','ts']].copy()

# Compute earliest anomaly timestamp per KPI; if KPI has no anomalies, record NaT
earliest = df_db002_kpis.groupby('kpi_name', as_index=False).agg(earliest_anomaly_timestamp=('ts','min'))

# Ensure all KPIs are present in final table (include missing KPI rows with NaT)
earliest_full = pd.DataFrame({'kpi_name': kpis_of_interest})
earliest_full = earliest_full.merge(earliest, on='kpi_name', how='left')
# Convert timestamps to tz-aware (they already are) and keep compact
earliest_full['earliest_anomaly_timestamp'] = pd.to_datetime(earliest_full['earliest_anomaly_timestamp']).dt.tz_convert(tz)

# Display compact outputs: A) db002_0430_anomalies (matching rows at 04:30) and B) earliest_full (per-KPI earliest timestamps)
db002_0430_anomalies, earliest_full

```
Out[7]:
```
```python
# Build a concise plain-English summary using kernel variables db002_0430_anomalies and earliest_full
try:
    db002_0430_anomalies
    earliest_full
except NameError:
    raise RuntimeError("Required kernel variables db002_0430_anomalies and earliest_full are missing.")

lines = []
lines.append("db_002 anomalies at 2020-05-25 04:30:00+08 (all anomalous KPIs at that minute):")
for r in db002_0430_anomalies.itertuples(index=False):
    # r has: timestamp, cmdb_id, kpi_name, value, flagged_side, used_threshold, threshold_value
    lines.append(f"- {r.kpi_name}: value={r.value} | side={r.flagged_side} | used_threshold={r.used_threshold} ({r.threshold_value})")

lines.append("")
lines.append("Earliest anomaly timestamp per requested KPI for db_002:")
for r in earliest_full.itertuples(index=False):
    ts = r.earliest_anomaly_timestamp
    ts_str = str(ts) if pd.notna(ts) else "no anomaly in window"
    lines.append(f"- {r.kpi_name}: {ts_str}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id               kpi_name       value flagged_side used_threshold  threshold_value
0  2020-05-25 04:30:00+08:00  db_002                    ACS    0.020000         high            p95         0.020000
1  2020-05-25 04:30:00+08:00  db_002                   AIOS    0.000000         high            p95         0.000000
2  2020-05-25 04:30:00+08:00  db_002                    AWS    0.000000          low            p05         0.000000
3  2020-05-25 04:30:00+08:00  db_002           Call_Per_Sec   16.700000          low            p05        16.800000
4  2020-05-25 04:30:00+08:00  db_002           Exec_Per_Sec   16.800000          low            p05        16.800000
5  2020-05-25 04:30:00+08:00  db_002                   Hang    1.000000         high            p95         1.000000
6  2020-05-25 04:30:00+08:00  db_002          Login_Per_Sec    0.400000         high            p95         0.400000
7  2020-05-25 04:30:00+08:00  db_002              MEM_Total  381.902264         high            p95       381.902264
8  2020-05-25 04:30:00+08:00  db_002           On_Off_State    1.000000         high            p95         1.000000
9  2020-05-25 04:30:00+08:00  db_002  Physical_Read_Per_Sec    0.000000          low            p05         0.000000
10 2020-05-25 04:30:00+08:00  db_002          Proc_Used_Pct    1.340000          low            p05         1.340000
11 2020-05-25 04:30:00+08:00  db_002     Proc_User_Used_Pct    1.340000          low            p05         1.340000
12 2020-05-25 04:30:00+08:00  db_002           Redo_Per_Sec  937.100000          low            p05       976.950000
13 2020-05-25 04:30:00+08:00  db_002               Row_Lock    0.000000         high            p95         0.000000
14 2020-05-25 04:30:00+08:00  db_002        SctRead_Per_Sec    0.000000         high            p95         0.000000
15 2020-05-25 04:30:00+08:00  db_002            Sess_Active    0.200000         high            p95         0.200000
16 2020-05-25 04:30:00+08:00  db_002    tnsping_result_time    0.000000          low            p05         0.000000,               kpi_name earliest_anomaly_timestamp
0                 Hang  2020-05-25 04:30:00+08:00
1         On_Off_State  2020-05-25 04:30:00+08:00
2  tnsping_result_time  2020-05-25 04:30:00+08:00
3        Login_Per_Sec  2020-05-25 04:30:00+08:00
4                  ACS  2020-05-25 04:30:00+08:00
5                  AWS  2020-05-25 04:30:00+08:00
6          Sess_Active  2020-05-25 04:30:00+08:00
7               DbTime  2020-05-25 04:37:00+08:00)```
```